# 精确 f696 名义 beta 守恒组成的合法目标空间

在原生 Qlib 冻结账户来源上，只统计 09:46 原执行日的 `SOXL−δ、SOXX+3δ、现金−2δ` 合法目标容量。事前冻结公式与支持门；不读取未来收益、拟合或运行候选账户。

In [1]:
import os,sys,hashlib,json
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd()
while not (root/'src').exists() and root!=root.parent:root=root.parent
sys.path.insert(0,str(root))
card=root/'research/stage1_f696_beta_neutral_rotation_support_20261025'
plan=json.loads((card/'plan.json').read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'result.json').exists()
prior=root/s['prior_headroom_card'];a=root/s['A0_card'];b=root/s['account_card']
assert sha(prior/'summary.json')==s['prior_headroom_summary_sha256']
assert sha(a/'summary.json')==s['A0_summary_sha256'] and sha(b/'summary.json')==s['account_summary_sha256']
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text());ra=json.loads((root/s['A0_readback']).read_text())
assert sa['nas_evidence']['sha256']==s['A0_NAS_sha256'] and ra['status']=='passed' and ra['archive_sha256']==s['A0_NAS_sha256'] and ra['all_artifact_hashes_match']
assert sb['nas_readback']['verified'] and sb['nas_readback']['sha256']==s['account_NAS_sha256']
for exp,rid,art,key in [('A0_experiment_id','A0_recorder_id','native/A_active_research_replay_0','A0_native_sha256'),('A0_experiment_id','A0_recorder_id','inputs/frame','A0_frame_sha256'),('A0_experiment_id','A0_recorder_id','inputs/recovery_spec','A0_recovery_spec_sha256')]:
 assert sha(a/'qlib_recorder'/s[exp]/s[rid]/'artifacts'/art)==s[key]
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['A0_experiment_id'],recorder_id=s['A0_recorder_id'])
 assert rec.status=='FINISHED'
 native=rec.load_object('native/A_active_research_replay_0');frame=rec.load_object('inputs/frame');recovery=rec.load_object('inputs/recovery_spec')
assert native['audit']['status']=='passed'
auth=native['authority'].reset_index(drop=True);targets=native['targets'].reset_index(drop=True);frame=frame.reset_index(drop=True)
for x in (auth,targets,frame):x['trade_date']=pd.to_datetime(x.trade_date).dt.normalize()
calendar=pd.DatetimeIndex(D.calendar(start_time='2023-01-04',end_time='2026-10-08',freq='day'))
assert len(calendar)==len(auth)==len(targets)==len(frame)==944
assert pd.DatetimeIndex(auth.trade_date).equals(calendar) and auth.trade_date.equals(targets.trade_date) and auth.trade_date.equals(frame.trade_date)
execute=auth.execute_open_target.to_numpy(bool);assert int(execute.sum())==797 and int((~execute).sum())==147
assert (float(recovery['minimum_cash_weight']),float(recovery['maximum_soxl_weight']))==(.15,.30)
w=targets[['soxl_weight','soxx_weight','cash_weight']].to_numpy(float)
assert np.isfinite(w).all() and np.max(np.abs(w.sum(axis=1)-1))<1e-9
assert (w[:,0]<=.30+1e-9).all() and (w[:,2]>=.15-1e-9).all()
cap=auth.current_beta_cap.to_numpy(float);assert np.isfinite(cap).all()
basebeta=3*w[:,0]+w[:,1];assert (basebeta<=cap+1e-8).all()
delta=np.where(execute,np.maximum(0.,np.minimum.reduce([w[:,0],np.maximum(0.,(w[:,2]-.15)/2),np.maximum(0.,(1-w[:,1])/3)])),0.)
candidate=np.column_stack([w[:,0]-delta,w[:,1]+3*delta,w[:,2]-2*delta])
assert (candidate>=-1e-9).all() and (candidate[:,0]<=.30+1e-9).all() and (candidate[:,1]<=1+1e-9).all() and (candidate[:,2]>=.15-1e-9).all()
assert np.max(np.abs(candidate.sum(axis=1)-1))<1e-9
assert np.max(np.abs(3*candidate[:,0]+candidate[:,1]-basebeta))<1e-9
assert (3*candidate[:,0]+candidate[:,1]<=cap+1e-8).all()
rawbeta=3*frame.soxl_weight.to_numpy(float)+frame.soxx_weight.to_numpy(float)
binding=rawbeta>cap+1e-9;positive=delta>1e-10
x=pd.DataFrame({'trade_date':calendar,'year':calendar.year,'quarter':calendar.quarter,'execute':execute,'binding':binding,'positive':positive,'delta_soxl':delta})
def cohort(g):
 e=g[g.execute];p=e[e.positive]
 return {'sessions':len(g),'execute':len(e),'HOLD':len(g)-len(e),'positive':len(p),'binding_positive':int(p.binding.sum()),'nonbinding_positive':int((~p.binding).sum()),'mean_delta_soxl_execute':float(e.delta_soxl.mean()) if len(e) else None,'max_delta_soxl':float(g.delta_soxl.max())}
all_days=cohort(x);annual=[{'year':int(y),'partial_endpoint':y in (2023,2026),**cohort(g)} for y,g in x.groupby('year')]
quarter=[{'year':int(y),'quarter':int(q),**cohort(g)} for (y,q),g in x.groupby(['year','quarter'])]
windows=[]
for i in range(819):
 g=x.iloc[i:i+126];v=cohort(g)
 windows.append({'first_trade':str(g.trade_date.iloc[0].date()),'last_trade':str(g.trade_date.iloc[-1].date()),'positive':v['positive'],'binding_positive':v['binding_positive'],'nonbinding_positive':v['nonbinding_positive']})
assert len(annual)==4 and len(quarter)==16 and len(windows)==819
passed=(all_days['positive']>=300 and all_days['binding_positive']>=100 and all_days['nonbinding_positive']>=100 and all(z['positive']>=50 for z in annual if z['year'] in (2024,2025)))
rolling_blob=(json.dumps(windows,ensure_ascii=False,separators=(',',':'))+'\n').encode()
rolling_path=card/'rolling126.json.gz'
with rolling_path.open('wb') as output:
 with gzip.GzipFile(fileobj=output,mode='wb',filename='',mtime=0,compresslevel=9) as archive:archive.write(rolling_blob)
rolling_sha=sha(rolling_path)
result={'decision':'legal_target_support_pass_needs_distinct_information_receiver' if passed else 'beta_neutral_rotation_target_support_failed','plan_sha256':sha(card/'plan.json'),'source_recorder_id':rec.id,'sessions':944,'rolling126_windows':819,'rolling126_file':rolling_path.name,'rolling126_sha256':rolling_sha,'rolling126_count':819,'max_nominal_beta_error':float(np.max(np.abs(3*candidate[:,0]+candidate[:,1]-basebeta))),'all':all_days,'annual':annual,'quarter':quarter,'support_gate_pass':passed,'new_model_fits':0,'new_native_accounts':0,'future_outcome_reads':0,'interpretation':'Source-target reachability only, no fee-net local value, own-state candidate account or SOXX performance.'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_id':rec.id,'source_NAS_readback_verified':True,'positive_EXECUTE':all_days['positive'],'binding_positive':all_days['binding_positive'],'nonbinding_positive':all_days['nonbinding_positive'],'support_gate_pass':passed,'rolling126_file':rolling_path.name,'rolling126_sha256':rolling_sha,'new_model_fits':0,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]},'limitations':['Read-only target-space accounting on original path, not a tradable counterfactual.','A static version on different parent previously failed risk/rolling gates; a new decision-time feature is required before any economic account.']}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':result['decision'],'all':all_days,'annual':annual},ensure_ascii=False))

{"decision": "beta_neutral_rotation_target_support_failed", "all": {"sessions": 944, "execute": 797, "HOLD": 147, "positive": 600, "binding_positive": 64, "nonbinding_positive": 536, "mean_delta_soxl_execute": 0.05774000250458028, "max_delta_soxl": 0.19163056165266132}, "annual": [{"year": 2023, "partial_endpoint": true, "sessions": 249, "execute": 200, "HOLD": 49, "positive": 160, "binding_positive": 12, "nonbinding_positive": 148, "mean_delta_soxl_execute": 0.06659533740987497, "max_delta_soxl": 0.18094016120211637}, {"year": 2024, "partial_endpoint": false, "sessions": 252, "execute": 220, "HOLD": 32, "positive": 188, "binding_positive": 18, "nonbinding_positive": 170, "mean_delta_soxl_execute": 0.07088219776140472, "max_delta_soxl": 0.1775472895357435}, {"year": 2025, "partial_endpoint": false, "sessions": 250, "execute": 206, "HOLD": 44, "positive": 150, "binding_positive": 18, "nonbinding_positive": 132, "mean_delta_soxl_execute": 0.053677655429869846, "max_delta_soxl": 0.1586114

# 精确 f696：名义 beta 守恒组成覆盖非受限现金层

只读已封存的精确 f696 A_0 0bp Tiered 原生 Qlib 账户和同源目标，核对944日、797个开盘EXECUTE、147个HOLD、16季度及全部819个连续126日窗口。诊断目标在原投影后按 `SOXL−δ、SOXX+3δ、现金−2δ` 变换，`δ` 只取 SOXL 原权重、现金15%底线及 SOXX 不超过100%所允许的最大非负值；HOLD不强制交易。全部目标权重合法，名义 beta 最大误差 **2.22e−16**，原自身风险上限不变。0未来收益读取、0拟合、0新账户。

797个原执行日中 **600日**有正的合法目标空间：原上限受限层 **64日**，非受限层 **536日**。2023/2024/2025/2026的正容量日分别为160/188/150/102，两个完整年度均超过50日；各年和季度计数在 `result.json`，819个窗口的容量计数压缩保存在 `rolling126.json.gz`。因预登记要求受限与非受限各至少100日，**单独此动作的双层支持门失败**，决定 `beta_neutral_rotation_target_support_failed`，不运行其单一路线账户。

这个负门也揭示了可研究的互补性：此前精确 f696 现金转 SOXL 诊断在受限/非受限原执行日有261/24日容量；本次 beta 守恒转换则为64/536日。两种动作都只是冻结原路径上的合法目标空间，不能相加为收益、不能按事后窗口选择，也不能证明联合策略可获利。旧名义beta组成账户使用其他母本且固定版本未过风险/滚动门，不能借本卡改称成功。下一步若继续，须先冻结一个**事前共同接收任务**，在训练期分别检验“保护现金还是部署”与“保留SOXL还是改配SOXX”的费用后局部价值及好区间损害，再在精确f696自身完整0bp账户测试双层联动；不能直接拼接两条静态动作。

来源身份、NAS收据和限定回读沿用 `stage1_domain_heads_20261018` 与 `stage1_original_0bp_signal_loss_chain_v2_20261023` 的封存Recorder。本卡没有新Recorder或行情快照，结果随Git保存。来源加载第一次因notebook内未设置仓库导入路径失败，0对象加载、0经济消耗；仅修复导入路径后完整执行，见 `attempt1_preflight_failure.json`。SOXX历史0/2/5bp门均未在本卡测试，Goal尚未达成。
